In [14]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

backend = BasicSimulator()

In [42]:
# function to generate ONE quantum random bit
# returns either 0 or 1

def quantum_random_bit():

  qc = QuantumCircuit(1,1)
  qc.h(0)
  qc.measure(0,0)
  compiled = transpile(qc, backend)
  result = backend.run(compiled, shots=1).result()
  counts = result.get_counts()
  bit = list(counts.keys())[0]
  return int(bit)

In [43]:
# generate multiple quantum random bits

def generate_quantum_bits(n):

    bits = []
    for _ in range(n):

        bits.append(
            quantum_random_bit()
        )
    return bits

In [44]:
# number of qubits
n = 20

# alice secret bits
alice_bits = generate_quantum_bits(n)

# alice random bases
# 0 = Z basis
# 1 = X basis
alice_bases = generate_quantum_bits(n)

print("alice bits:")
print(alice_bits)

print("\nalice bases:")
print(alice_bases)

alice bits:
[0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0]

alice bases:
[1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1]


In [45]:
# function to encode qubits

def encode_qubit(bit, basis):

    qc = QuantumCircuit(1,1)

    # if bit is 1 apply X gate
    if bit == 1:
        qc.x(0)

    # if basis is X basis apply Hadamard gate
    if basis == 1:
        qc.h(0)

    return qc

# store all encoded qubits
alice_qubits = []

# encode each bit
for bit, basis in zip(alice_bits, alice_bases):

    alice_qubits.append(
        encode_qubit(bit, basis)
    )

In [46]:
# evan intercepts
# evan chooses random bases
evan_bases = generate_quantum_bits(n)

evan_results = []

# modified qubits after evan measures
modified_qubits = []

# evan intercepts qubits
for qc, basis in zip(alice_qubits, evan_bases):

    intercepted = qc.copy()

    # evan uses X basis
    if basis == 1:
        intercepted.h(0)

    # measure qubit
    intercepted.measure(0,0)

    compiled = transpile(intercepted, backend)

    result = backend.run(compiled, shots=1).result()

    counts = result.get_counts()

    evan_bit = int(
        list(counts.keys())[0]
    )

    evan_results.append(evan_bit)

    # evan resends qubit
    resend = QuantumCircuit(1,1)

    if evan_bit == 1:
        resend.x(0)

    if basis == 1:
        resend.h(0)

    modified_qubits.append(resend)

print("evan bases:")
print(evan_bases)

print("\nevan results:")
print(evan_results)


evan bases:
[1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0]

evan results:
[0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1]


In [47]:
# bob chooses random bases

bob_bases = generate_quantum_bits(n)

print("bob bases:")
print(bob_bases)

bob bases:
[1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1]


In [48]:
# Function for Bob to measure qubits

def measure_qubit(qc, basis):

    # if bob uses X basis
    # apply Hadamard before measuring
    if basis == 1:
        qc.h(0)

    qc.measure(0,0)
    compiled = transpile(qc, backend)
    result = backend.run(compiled, shots=1).result()
    counts = result.get_counts()
    bit = list(counts.keys())[0]
    return int(bit)

# bob measurement results
bob_results = []

# measure each qubit
for qc, basis in zip(modified_qubits, bob_bases):

    # copy circuit so original is unchanged
    new_qc = qc.copy()

    result = measure_qubit(new_qc, basis)

    bob_results.append(result)

print("bob results:")
print(bob_results)

bob results:
[0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0]


In [49]:
# shared keys
shared_key_alice = []
shared_key_bob = []

# compare bases
for i in range(n):

    # keep bits only if bases match
    if alice_bases[i] == bob_bases[i]:

        shared_key_alice.append(
            alice_bits[i]
        )

        shared_key_bob.append(
            bob_results[i]
        )

print("shared key (alice):")
print(shared_key_alice)

print("\nshared key (bob):")
print(shared_key_bob)

shared key (alice):
[0, 1, 1, 1, 1, 1, 0, 1, 0]

shared key (bob):
[0, 0, 0, 1, 1, 0, 0, 1, 0]


In [50]:
errors = 0

for a, b in zip(shared_key_alice, shared_key_bob):

    if a != b:
        errors += 1

error_rate = errors / len(shared_key_alice)

print("number of errors:", errors)

print("error rate:", error_rate)

# Threshold for attack detection
threshold = 0.2

if error_rate > threshold:

    print("\nATTACKER DETECTED")

else:

    print("\nno attacker detected")

number of errors: 3
error rate: 0.3333333333333333

ATTACKER DETECTED
